# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [57]:
# Reset to a safe directory BEFORE deleting the old repository
%cd /content

!rm -rf /content/flyrank-ml-internship

!git clone https://github.com/Imvixh/flyrank-ml-internship.git /content/flyrank-ml-internship

from pathlib import Path

ROOT = Path("/content/flyrank-ml-internship")
DATA = ROOT / "data/raw/content_refresh_anonymized.csv"

print("Repository exists:", ROOT.exists())
print("Dataset exists:", DATA.exists())
print("Dataset path:", DATA)

/content
Cloning into '/content/flyrank-ml-internship'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 120 (delta 36), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 1.89 MiB | 4.50 MiB/s, done.
Resolving deltas: 100% (36/36), done.
Repository exists: True
Dataset exists: True
Dataset path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [58]:
%cd /content/flyrank-ml-internship


/content/flyrank-ml-internship


In [60]:
%cd /content/flyrank-ml-internship

!python scripts/run_all.py

/content/flyrank-ml-internship

▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/flyrank-ml-internship/outputs/refresh_queue.csv
Wr

In [61]:
import pandas as pd

QUEUE = ROOT / "outputs/refresh_queue.csv"

print("Queue exists:", QUEUE.exists())

queue = pd.read_csv(QUEUE)

print("Rows:", len(queue))
print("\nActions:")
display(queue["suggested_action"].value_counts())

print("\nTop 10:")
display(
    queue[
        [
            "final_rank",
            "final_refresh_score",
            "confidence",
            "suggested_action",
            "final_reason_codes",
            "impressions_90d",
            "sessions_90d",
            "trend_direction",
        ]
    ].head(10)
)

Queue exists: True
Rows: 30000

Actions:


,count
suggested_action,
monitor,13083
refresh,8188
refresh_and_review_ctr,6654
refresh_and_review_engagement,1993
expand_and_refresh,82



Top 10:


,final_rank,final_refresh_score,confidence,suggested_action,final_reason_codes,impressions_90d,sessions_90d,trend_direction
0,1,81.734212,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,12834,66,down
1,2,81.603243,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2498,9,down
2,3,81.544618,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,8064,23,down
3,4,81.169731,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,13790,27,down
4,5,80.957565,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,3393,5,down
5,6,80.798090,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,5811,14,down
6,7,80.650656,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1622,20,down
7,8,80.432641,medium,refresh,declining_with_demand|model_decline_risk|visib...,4366,8,down
8,9,80.428403,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1597,5,down
9,10,80.428243,medium,refresh,declining_with_demand|model_decline_risk|visib...,3867,5,down


### Ranked actions and reason codes

The validated model output is converted into a ranked content-action queue. Higher
`final_refresh_score` items are reviewed first. The queue combines model probability
with the existing baseline refresh score, so the recommendation is not based on the
model alone.

The main actions are:

- **refresh** — evidence suggests the content may need updating.
- **refresh_and_review_ctr** — the page has meaningful impressions but relatively low CTR, so the reviewer should inspect title/snippet and search-result alignment.
- **refresh_and_review_engagement** — the page has enough sessions but weaker engagement signals, so the reviewer should inspect content usefulness and page experience.
- **expand_and_refresh** — the page appears thin and should be considered for substantive expansion before refresh.
- **monitor** — evidence is not strong enough to justify an immediate refresh.

Reason codes explain why an item entered the queue. Examples include `declining_with_demand`,
`low_ctr_visible_page`, `low_engagement_visible_page`, `model_decline_risk`,
`visible_model_opportunity`, and review-candidate flags.

These recommendations are decision-support signals, not automatic publishing decisions.
A human should inspect the underlying page and editorial context before taking action.

In [62]:
# Section 1 — ranked actions + reason codes

print("===== ML-10 RANKED ACTION QUEUE =====")
print(f"Rows in queue: {len(queue):,}")

print("\nAction distribution:")
display(queue["suggested_action"].value_counts())

print("\nTop 10 ranked recommendations:")
display(
    queue[
        [
            "final_rank",
            "final_refresh_score",
            "best_model_probability",
            "confidence",
            "suggested_action",
            "final_reason_codes",
            "impressions_90d",
            "sessions_90d",
            "trend_direction",
        ]
    ].head(10)
)

===== ML-10 RANKED ACTION QUEUE =====
Rows in queue: 30,000

Action distribution:


,count
suggested_action,
monitor,13083
refresh,8188
refresh_and_review_ctr,6654
refresh_and_review_engagement,1993
expand_and_refresh,82



Top 10 ranked recommendations:


,final_rank,final_refresh_score,best_model_probability,confidence,suggested_action,final_reason_codes,impressions_90d,sessions_90d,trend_direction
0,1,81.734212,0.783472,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,12834,66,down
1,2,81.603243,0.849842,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2498,9,down
2,3,81.544618,0.789490,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,8064,23,down
3,4,81.169731,0.776297,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,13790,27,down
4,5,80.957565,0.816010,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,3393,5,down
5,6,80.798090,0.796332,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,5811,14,down
6,7,80.650656,0.846499,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1622,20,down
7,8,80.432641,0.827496,medium,refresh,declining_with_demand|model_decline_risk|visib...,4366,8,down
8,9,80.428403,0.844030,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1597,5,down
9,10,80.428243,0.845325,medium,refresh,declining_with_demand|model_decline_risk|visib...,3867,5,down


### Intended use and limits

The action playbook is intended to help human reviewers prioritize which content
should be inspected first. It is a decision-support tool, not an autonomous
content-management system.

The ranked queue is useful for identifying pages with stronger evidence of
decline, weak search performance, weak engagement, or content-refresh opportunity.

The recommendations should not be treated as proof that a page is objectively
bad or that refreshing it will cause traffic or rankings to increase.

Important limits:

- The model is trained on historical observed data.
- Model scores represent estimated risk/opportunity, not causal effects.
- Search-engine ranking behavior cannot be inferred directly from this model.
- Low CTR does not automatically mean that the content itself is poor.
- A declining trend can have external causes that are not represented in the data.
- Small-volume pages should be treated cautiously because their metrics are less stable.
- Recommendations require human review before any content, metadata, or publishing action.

The safest use is therefore: **rank → inspect → validate context → decide → act**.

In [63]:
# Section 2 — intended use and limits

print("===== INTENDED USE =====")

print("""
Purpose:
Prioritize content for human review using observable model and search-performance
signals.

This is a decision-support system, not an autonomous publishing system.

Human review is required before:
- changing content
- changing metadata
- merging or pruning pages
- publishing updates
- making strategic SEO decisions
""")

print("\nKey limitations:")
print("1. Model predictions are directional, not causal.")
print("2. Search-engine ranking behavior is not proven by this model.")
print("3. Low CTR does not automatically mean poor content.")
print("4. Small-volume pages require additional caution.")
print("5. External factors may explain observed declines.")

===== INTENDED USE =====

Purpose:
Prioritize content for human review using observable model and search-performance
signals.

This is a decision-support system, not an autonomous publishing system.

Human review is required before:
- changing content
- changing metadata
- merging or pruning pages
- publishing updates
- making strategic SEO decisions


Key limitations:
1. Model predictions are directional, not causal.
2. Search-engine ranking behavior is not proven by this model.
3. Low CTR does not automatically mean poor content.
4. Small-volume pages require additional caution.
5. External factors may explain observed declines.


### Human review and no-go list

Every recommended action must be reviewed by a human before execution.

The reviewer should check the page itself, search intent, current SERP context,
recent business changes, content quality, and whether the observed signal is
large enough to justify action.

#### Human review checklist

1. Confirm that the page is still relevant to the target search intent.
2. Inspect the current title and search-result snippet.
3. Check whether impressions and traffic are sufficient to make the signal meaningful.
4. Check for recent updates, migrations, redirects, or technical changes.
5. Compare the recommendation with the actual page and business context.
6. Record the final human decision before making a production change.

#### No-go cases

Do not automatically act when:

- the page has very little search visibility or traffic;
- the recommendation conflicts with clear business or editorial requirements;
- the page is affected by a known technical or tracking problem;
- the observed decline is likely caused by seasonality or an external event;
- the content is legally, medically, financially, or otherwise high-stakes without appropriate expert review;
- the model recommendation is based on incomplete or suspicious data;
- there is insufficient evidence to distinguish a real opportunity from normal noise.

The model should prioritize investigation, not replace human judgment.

In [64]:
print("===== HUMAN REVIEW / NO-GO CHECK =====")

no_go_rules = [
    "Very low visibility or traffic",
    "Known tracking or technical problem",
    "Strong seasonality or external event",
    "Business/editorial conflict",
    "High-stakes content requiring expert review",
    "Incomplete or suspicious data",
    "Insufficient evidence for a reliable decision",
]

for i, rule in enumerate(no_go_rules, 1):
    print(f"{i}. {rule}")

print("\nRESULT: Human review required before production action.")

===== HUMAN REVIEW / NO-GO CHECK =====
1. Very low visibility or traffic
2. Known tracking or technical problem
3. Strong seasonality or external event
4. Business/editorial conflict
5. High-stakes content requiring expert review
6. Incomplete or suspicious data
7. Insufficient evidence for a reliable decision

RESULT: Human review required before production action.


### Monitoring and retrain triggers

The action playbook should be monitored after deployment because the relationship
between the input signals and observed outcomes can change over time.

The following conditions should trigger investigation:

- **Data drift:** important feature distributions change substantially from the
  training data.
- **Performance drift:** Precision@50 or another agreed evaluation metric falls
  materially below the validation result.
- **Prediction drift:** the proportion of high-risk or high-priority recommendations
  changes unexpectedly.
- **Outcome drift:** the observed declining rate changes substantially over time.
- **Feature quality problems:** missing values, invalid values, tracking changes, or
  unexpected changes in metric definitions appear.
- **Business/process changes:** major changes to the website, search strategy,
  measurement system, or content workflow make the historical training data less
  representative.

### Retraining rule

Retraining should not happen simply because time has passed. It should be triggered
when monitoring shows that the model is no longer reliable for the intended decision.

Before retraining, the data and label definitions should be rechecked for leakage,
consistency, and comparability with the previous evaluation.

After retraining, the new model should be compared with the existing model and the
Week-4 baseline on the same appropriate validation design before being adopted.

The model should be rolled back or removed from the workflow if its performance or
data quality becomes unreliable.

In [65]:
# Section 4 — monitoring and retrain triggers

print("===== MONITORING / RETRAIN TRIGGERS =====")

triggers = [
    "Feature/data distributions drift materially",
    "Precision@50 falls materially below validation performance",
    "High-priority prediction volume changes unexpectedly",
    "Observed declining rate changes substantially",
    "Missing, invalid, or inconsistent feature values appear",
    "Business, website, tracking, or measurement changes occur",
]

for i, trigger in enumerate(triggers, 1):
    print(f"{i}. {trigger}")

print("\n===== RETRAINING POLICY =====")
print("Retrain only after evidence of model/data drift or reduced reliability.")
print("Recheck leakage and label consistency before retraining.")
print("Compare the new model against the existing model and baseline.")
print("Do not deploy a retrained model unless validation supports the change.")

===== MONITORING / RETRAIN TRIGGERS =====
1. Feature/data distributions drift materially
2. Precision@50 falls materially below validation performance
3. High-priority prediction volume changes unexpectedly
4. Observed declining rate changes substantially
5. Missing, invalid, or inconsistent feature values appear
6. Business, website, tracking, or measurement changes occur

===== RETRAINING POLICY =====
Retrain only after evidence of model/data drift or reduced reliability.
Recheck leakage and label consistency before retraining.
Compare the new model against the existing model and baseline.
Do not deploy a retrained model unless validation supports the change.


### Exports for the paper

The ranked action queue generated by this notebook is the main output for the
research paper. It provides a reproducible list of content items, their scores,
confidence, recommended actions, and reason codes.

The export should be regenerated from the notebook rather than manually edited.
This keeps the paper's recommendations traceable to the underlying model output.

The exported queue is intended for analysis and human review. It should not be
treated as an automated production action list.

In [66]:
# Section 5 — export the ranked queue for the paper

from pathlib import Path

OUT = ROOT / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

export_path = OUT / "ml10_ranked_action_queue.csv"

queue.to_csv(export_path, index=False)

print("===== ML-10 EXPORT =====")
print("Export path:", export_path)
print("Rows exported:", len(queue))
print("File exists:", export_path.exists())

===== ML-10 EXPORT =====
Export path: /content/flyrank-ml-internship/work/outputs/ml10_ranked_action_queue.csv
Rows exported: 30000
File exists: True


## Self-check

Before you submit, confirm each line honestly:

- ☑ Every section above is filled — markdown thinking AND the code that backs it
- ☑ The notebook runs top to bottom with no errors (Runtime → Run all)
- ☑ The ranked action queue is generated and exported successfully
- ☑ No client names, URLs, or private queries anywhere
- ☑ My claims use careful words: observed, measured, directional, decision-support
- ☑ Human review and no-go cases are clearly stated
- ☑ Monitoring and retraining triggers are documented
- ☑ Committed to my repo under `work/notebooks/` — then submit my repo URL on the card.

**Done.**